# 16 — Subtype results aggregation

Mirror of notebook 11 for the lung_aca vs lung_scc task. This is the experiment where models actually have room to differ — expect AUCs in the 0.85–0.97 range rather than the saturated 1.0 of the malignant-vs-benign task.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from sklearn.metrics import (
    confusion_matrix,
    precision_recall_curve,
    roc_curve,
    auc,
    average_precision_score,
)

from utils.training import load_fold_results
from utils.metrics import aggregate_folds, per_fold_metric_list
from utils.stats import format_results_table, pairwise_wilcoxon

with open('../configs/histology_subtype.yaml') as f:
    cfg = yaml.safe_load(f)

results_dir = Path('..') / cfg['paths']['results']
figures_dir = Path('..') / cfg['paths']['figures']
figures_dir.mkdir(parents=True, exist_ok=True)

MODELS = ['SVM', 'RF', 'KNN', 'ResNet50_pretrained', 'ResNet50_scratch']
all_folds = {name: load_fold_results(results_dir / f'subtype_{name}.json') for name in MODELS}

### Threshold rescue for KNN

Youden's J on the original training fold returned `threshold_calibrated = 1.0` for KNN, because distance-weighted KNN produces a near-perfectly separable training fold with many probability ties at exactly 1.0 — Youden's J is maximised on a flat plateau and the saved code picked the highest threshold in the plateau. We patched [utils/metrics.py](../utils/metrics.py) so future runs pick the threshold **closest to 0.5 among all that achieve max J**.

For the already-saved results, instead of re-fitting on test data (which would amount to test-set threshold tuning for the other models), we **fall back to the default-threshold metrics** (`threshold = 0.5`, already pre-computed and stored as `metrics_default`) only for the folds where the saved threshold is degenerate (≥ 0.99 or ≤ 0.01). Valid Youden thresholds from the other models stay untouched.

In [ ]:
from utils.metrics import find_optimal_threshold, compute_metrics

# Only rescue folds where the saved Youden threshold is degenerate (≥0.99 or ≤0.01) —
# i.e. it collapsed onto an extreme due to discrete-probability ties on the training
# fold. For those folds we substitute the default-threshold metrics (threshold=0.5,
# already pre-computed and stored in `metrics_default`). Valid Youden thresholds from
# the other models are left untouched, so no test-set threshold tuning is introduced.
rescued = []
for name, folds in all_folds.items():
    for f in folds:
        if f.threshold_calibrated >= 0.99 or f.threshold_calibrated <= 0.01:
            rescued.append((name, f.fold, f.threshold_calibrated))
            f.threshold_calibrated = 0.5
            f.metrics_calibrated = dict(f.metrics_default)

if rescued:
    print('Folds rescued (degenerate threshold → 0.5 / metrics_default):')
    for name, fold, old in rescued:
        print(f'  {name} fold {fold}: old thr = {old:.3f}')
else:
    print('No degenerate thresholds detected — nothing rescued.')

In [2]:
agg = {name: aggregate_folds([f.metrics_calibrated for f in folds]) for name, folds in all_folds.items()}
table = format_results_table(agg)
table.to_csv(results_dir / 'subtype_summary_calibrated.csv')
table

,accuracy,sensitivity,specificity,precision,f1,balanced_accuracy,auc_roc,pr_auc
model,,,,,,,,
SVM,0.936 ± 0.056,0.873 ± 0.113,0.999 ± 0.002,0.999 ± 0.002,0.929 ± 0.065,0.936 ± 0.056,0.999 ± 0.001,0.999 ± 0.002
RF,0.964 ± 0.014,0.936 ± 0.024,0.993 ± 0.010,0.993 ± 0.010,0.963 ± 0.015,0.964 ± 0.014,0.998 ± 0.002,0.997 ± 0.005
KNN,0.902 ± 0.030,0.805 ± 0.061,0.999 ± 0.002,0.999 ± 0.003,0.890 ± 0.037,0.902 ± 0.030,0.996 ± 0.002,0.996 ± 0.003
ResNet50_pretrained,0.993 ± 0.005,0.990 ± 0.008,0.995 ± 0.006,0.995 ± 0.006,0.992 ± 0.005,0.993 ± 0.005,1.000 ± 0.000,1.000 ± 0.000
ResNet50_scratch,0.950 ± 0.010,0.949 ± 0.021,0.951 ± 0.020,0.951 ± 0.018,0.950 ± 0.010,0.950 ± 0.010,0.993 ± 0.004,0.993 ± 0.004


In [3]:
rows = []
for name, folds in all_folds.items():
    for f in folds:
        rows.append({'model': name, 'fold': f.fold, **f.metrics_calibrated})
per_fold_df = pd.DataFrame(rows)
per_fold_df.to_csv(results_dir / 'subtype_per_fold.csv', index=False)
per_fold_df

,model,fold,accuracy,sensitivity,specificity,precision,f1,balanced_accuracy,tp,tn,fp,fn,threshold,auc_roc,pr_auc
0,SVM,0,0.9750,0.950,1.000,1.000000,0.974359,0.9750,190,200,0,10,0.928031,0.999375,0.999375
1,SVM,1,0.9575,0.920,0.995,0.994595,0.955844,0.9575,184,199,1,16,0.959634,0.996775,0.995996
2,SVM,2,0.8750,0.750,1.000,1.000000,0.857143,0.8750,150,200,0,50,0.996270,0.998575,0.998570
3,SVM,3,0.9950,0.990,1.000,1.000000,0.994975,0.9950,198,200,0,2,0.870492,1.000000,1.000000
4,SVM,4,0.8775,0.755,1.000,1.000000,0.860399,0.8775,151,200,0,49,0.995841,0.999450,0.999460
5,RF,0,0.9600,0.920,1.000,1.000000,0.958333,0.9600,184,200,0,16,0.722399,0.998125,0.998158
6,RF,1,0.9625,0.945,0.980,0.979275,0.961832,0.9625,189,196,4,11,0.722000,0.994188,0.988754
7,RF,2,0.9450,0.905,0.985,0.983696,0.942708,0.9450,181,197,3,19,0.716000,0.996912,0.996796
8,RF,3,0.9725,0.945,1.000,1.000000,0.971722,0.9725,189,200,0,11,0.735000,0.999412,0.999403
9,RF,4,0.9825,0.965,1.000,1.000000,0.982188,0.9825,193,200,0,7,0.715000,0.999875,0.999874


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, folds in all_folds.items():
    y_true = np.concatenate([np.array(f.y_true) for f in folds])
    y_proba = np.concatenate([np.array(f.y_proba) for f in folds])
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc(fpr, tpr):.3f})')
    p, r, _ = precision_recall_curve(y_true, y_proba)
    axes[1].plot(r, p, label=f'{name} (AP={average_precision_score(y_true, y_proba):.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].set_title('ROC — out-of-fold (subtype)'); axes[0].legend()
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision'); axes[1].set_title('PR — out-of-fold (subtype)'); axes[1].legend()
plt.tight_layout(); plt.savefig(figures_dir / 'subtype_roc_pr.png', dpi=140); plt.show()

/var/folders/v_/vcl9p7t96w78vd6w2ddjyw2w0000gn/T/ipykernel_25158/2186848283.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(figures_dir / 'subtype_roc_pr.png', dpi=140); plt.show()


In [5]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(4 * len(MODELS), 4))
for ax, (name, folds) in zip(axes, all_folds.items()):
    y_true = np.concatenate([np.array(f.y_true) for f in folds])
    y_proba = np.concatenate([np.array(f.y_proba) for f in folds])
    thr = float(np.mean([f.threshold_calibrated for f in folds]))
    y_pred = (y_proba >= thr).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['scc', 'aca'], yticklabels=['scc', 'aca'])
    ax.set_title(f'{name}\nthr={thr:.2f}'); ax.set_xlabel('predicted'); ax.set_ylabel('actual')
plt.tight_layout(); plt.savefig(figures_dir / 'subtype_confusion.png', dpi=140); plt.show()

/var/folders/v_/vcl9p7t96w78vd6w2ddjyw2w0000gn/T/ipykernel_25158/2105402712.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(figures_dir / 'subtype_confusion.png', dpi=140); plt.show()


In [6]:
metrics = ['accuracy', 'sensitivity', 'specificity', 'precision', 'f1', 'balanced_accuracy', 'auc_roc', 'pr_auc']
means = pd.DataFrame({m: {name: agg[name][m]['mean'] for name in MODELS} for m in metrics})
stds = pd.DataFrame({m: {name: agg[name][m]['std'] for name in MODELS} for m in metrics})
ax = means.T.plot(kind='bar', yerr=stds.T, figsize=(14, 5), capsize=2)
ax.set_ylim(0, 1.05); ax.set_ylabel('score'); ax.set_title('LC25000 subtype — model comparison (5-fold CV, calibrated)')
ax.legend(loc='lower right'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(figures_dir / 'subtype_bars.png', dpi=140); plt.show()

/var/folders/v_/vcl9p7t96w78vd6w2ddjyw2w0000gn/T/ipykernel_25158/1847746505.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(figures_dir / 'subtype_bars.png', dpi=140); plt.show()


In [7]:
auc_by_model = {name: per_fold_metric_list([f.metrics_calibrated for f in folds], 'auc_roc')
                for name, folds in all_folds.items()}
p_matrix = pairwise_wilcoxon(auc_by_model)
p_matrix.to_csv(results_dir / 'subtype_wilcoxon_auc.csv')
p_matrix

,SVM,RF,KNN,ResNet50_pretrained,ResNet50_scratch
SVM,1.0000,0.1250,0.0625,0.1250,0.0625
RF,0.1250,1.0000,0.1875,0.0625,0.0625
KNN,0.0625,0.1875,1.0000,0.0625,0.4375
ResNet50_pretrained,0.1250,0.0625,0.0625,1.0000,0.0625
ResNet50_scratch,0.0625,0.0625,0.4375,0.0625,1.0000


## Comparison with the malignant-vs-benign task

Side-by-side AUC means from notebook 11 (binary) and this notebook (subtype). If the binary AUCs are pinned at ~1.0 while the subtype AUCs spread out, that confirms the original task was saturated and the subtype task is the one that actually ranks models.

In [8]:
binary_folds = {name: load_fold_results(results_dir / f'histology_{name}.json') for name in MODELS}
binary_agg = {name: aggregate_folds([f.metrics_calibrated for f in folds]) for name, folds in binary_folds.items()}

comparison = pd.DataFrame({
    'binary AUC (mean)':   [binary_agg[m]['auc_roc']['mean'] for m in MODELS],
    'binary AUC (std)':    [binary_agg[m]['auc_roc']['std']  for m in MODELS],
    'subtype AUC (mean)':  [agg[m]['auc_roc']['mean']        for m in MODELS],
    'subtype AUC (std)':   [agg[m]['auc_roc']['std']         for m in MODELS],
}, index=MODELS)
comparison.to_csv(results_dir / 'task_comparison_auc.csv')
comparison

,binary AUC (mean),binary AUC (std),subtype AUC (mean),subtype AUC (std)
SVM,1.000000,0.000000,0.998835,0.001259
RF,1.000000,0.000000,0.997703,0.002280
KNN,1.000000,0.000000,0.995887,0.001976
ResNet50_pretrained,1.000000,0.000000,0.999855,0.000130
ResNet50_scratch,0.999925,0.000061,0.992575,0.003670


## Why even the subtype task saturates — t-SNE on radiomic features

Project the 208-dim color+GLCM+LBP+HED feature vector to 2D with t-SNE on a 1500-per-class subsample. If aca and scc form mostly disjoint clusters in feature space, no classifier needs to do anything sophisticated — even KNN can hit AUC ≈ 1.0. This is the visual confirmation that LC25000 is intrinsically easy, separate from any leakage concerns.

In [ ]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

# Reuse the binary task's features.npz — it covers all three lung classes,
# from which we filter aca + scc to match the subtype task.
binary_cache = results_dir / 'histology_cache' / 'features.npz'
feat = np.load(binary_cache, allow_pickle=True)
paths = feat['path']
is_aca = np.array(['/lung_aca/' in str(p) for p in paths])
is_scc = np.array(['/lung_scc/' in str(p) for p in paths])
is_n   = np.array(['/lung_n/'   in str(p) for p in paths])

rng = np.random.default_rng(cfg['seed'])
def _sample(mask, n=1500):
    idx = np.where(mask)[0]
    return rng.choice(idx, size=min(n, len(idx)), replace=False)

idx_aca = _sample(is_aca); idx_scc = _sample(is_scc); idx_n = _sample(is_n)
idx_all = np.concatenate([idx_aca, idx_scc, idx_n])
labels_str = np.array(['aca']*len(idx_aca) + ['scc']*len(idx_scc) + ['n']*len(idx_n))
X_sub = StandardScaler().fit_transform(feat['X'][idx_all])

emb = TSNE(n_components=2, perplexity=40, init='pca', learning_rate='auto',
           random_state=cfg['seed']).fit_transform(X_sub)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
palette = {'aca': '#d62728', 'scc': '#1f77b4', 'n': '#2ca02c'}

for cls in ['aca', 'scc']:
    m = (labels_str == cls)
    axes[0].scatter(emb[m, 0], emb[m, 1], s=6, alpha=0.6, c=palette[cls], label=cls)
axes[0].set_title('Subtype task: lung_aca vs lung_scc'); axes[0].legend(); axes[0].set_xticks([]); axes[0].set_yticks([])

for cls in ['aca', 'scc', 'n']:
    m = (labels_str == cls)
    axes[1].scatter(emb[m, 0], emb[m, 1], s=6, alpha=0.6, c=palette[cls], label=cls)
axes[1].set_title('All three lung classes (binary task: n vs aca+scc)'); axes[1].legend(); axes[1].set_xticks([]); axes[1].set_yticks([])

plt.tight_layout(); plt.savefig(figures_dir / 'subtype_tsne.png', dpi=140); plt.show()